# Test 01 — the ellipse tube

This test consisted of an elliptical tube 500 mm in length, printed in two halves, rods glued in, laid on two supports and
loaded at mid-span. **It held 80 kgf with no failure.** 

Later it took an impact of unknown magnitued (a very scientific "jumping" in the middle section) and the **rods broke in compression, just outside the joints**. 

![Cross-section of the ellipse tube](../../imagenes/ellipse-cross-section.png)


**How it failed, also a learning**, and promising, given that it seemed that the rods broke with a clean cut (compression side), right after the joined section ends (right in the middle where the load was higher, and the step-down from 2 continous rod pairs into 1)

![Cross-seciton of the ellipse after failure](../../imagenes/ellipse_post_failure.jpg)

## Geometry

| | as built |
|---|---|
| section | ellipse, 100 x 30 mm (semi-axes 50 x 15) |
| length | 500 mm, two 250 mm printed sections bonded at mid-span |
| print orientation | standing on end ("spanwise"). The tube axis is the build Z |
| wall | **1 perimeter, 0.42 mm** |
| infill | **7 %, cubic** |
| rods | 8 off, 3 mm pultruded carbon, 250 mm each |
| rod layout | 2 pairs at x = +/-20 mm, upper and lower centres 19.5 mm apart |
| rod runs | two per line, lapping 50 mm across mid-span — 4 rods in a section, 8 in the lap |
| bells | one per rod line, ~10 mm² of printed material each |
| test | 3-point bend, supports 450 mm apart, 80 kgf = 785 N at mid-span |

**The bell.** Each bore is wrapped in a bell-shaped region of denser print that flares out
from the bore toward the nearest skin. In section it is the `wing_builder` bell profile,
$y = h\,|2x/w|^{3.5}$, 10 mm across the mouth and 7 mm tall. Its narrow end sits 3 mm inside
the rod centre on the skin normal, and its mouth opens onto the skin. It runs the full
length of the rod, so it is a prism along the tube. It gives the rod a solid load path into the wall: shear
flows from the bore wall, through the bell, into the skin, and never through the 7 % fill.

In the bending sum it is **10 mm² of wall-modulus plastic per rod line**, placed at the
centroid of that outline after clipping to the tube interior and cutting out the bore. The
outline itself covers more than 10 mm². Printed at partial density, it holds about 10 mm² of
solid plastic, so the fill is removed over the whole outline and only the 10 mm² added back.
One bell per line, also through the lap. The tube was not built with `wing_builder`, so
the outline's position is taken from its defaults.

Still not counted: the sleeve at the bonded joint. It adds stiffness, so leaving it out errs
the right way. The bore is taken as the rod plus a 0.4 mm glue gap, and the infill inside it
is removed.

In [1]:
from __future__ import annotations
import math
from dataclasses import dataclass

import numpy as np

G = 9.81          # m/s^2.  Lengths mm, forces N, stresses MPa throughout.


@dataclass
class Tube:
    """The article as built."""
    a: float = 50.0            # mm  semi-major, horizontal
    b: float = 15.0            # mm  semi-minor, vertical
    length: float = 500.0      # mm  two bonded 250 mm sections
    wall_t: float = 0.42       # mm  ONE perimeter
    rod_d: float = 3.0         # mm
    glue_gap: float = 0.4      # mm  bore diameter - rod diameter
    rod_x: float = 20.0        # mm  |x| of each rod pair
    rod_dy: float = 19.5       # mm  lower rod centre to upper rod centre
    rod_len: float = 250.0     # mm
    rod_overlap: float = 50.0  # mm  the two rods on a line lap across mid-span
    bell_area: float = 10.0    # mm^2 printed material in one bell, solid-equivalent
    bell_width: float = 10.0   # mm  mouth, at the skin
    bell_height: float = 7.0   # mm  along the skin normal
    bell_offset: float = -3.0  # mm  narrow end, from the rod centre along the outward normal
    bell_shape: float = 3.5    #     exponent of the flare
    n_poly: int = 720

    @property
    def r_rod(self) -> float:  return self.rod_d / 2
    @property
    def r_bore(self) -> float: return (self.rod_d + self.glue_gap) / 2
    @property
    def rod_y(self) -> float:  return self.rod_dy / 2

    def outer(self) -> np.ndarray:
        t = np.linspace(0, 2 * math.pi, self.n_poly, endpoint=False)
        return np.column_stack([self.a * np.cos(t), self.b * np.sin(t)])

    def inner(self) -> np.ndarray:
        """The outer ellipse pulled in by one wall along its local normal."""
        t = np.linspace(0, 2 * math.pi, self.n_poly, endpoint=False)
        nx, ny = self.b * np.cos(t), self.a * np.sin(t)
        n = np.hypot(nx, ny)
        return np.column_stack([self.a * np.cos(t) - self.wall_t * nx / n,
                                self.b * np.sin(t) - self.wall_t * ny / n])

    def rod_centres(self) -> list:
        return [(sx * self.rod_x, sy * self.rod_y) for sx in (-1, 1) for sy in (1, -1)]

    def rod_runs(self) -> list:
        """(z_start, z_end) of every physical rod. Two per line, lapping at mid-span."""
        mid, out = self.length / 2, []
        for _ in self.rod_centres():
            out.append((mid + self.rod_overlap / 2 - self.rod_len, mid + self.rod_overlap / 2))
            out.append((mid - self.rod_overlap / 2, mid - self.rod_overlap / 2 + self.rod_len))
        return out

    def n_rods_at(self, z: float) -> int:
        return sum(1 for z0, z1 in self.rod_runs() if z0 <= z <= z1)


tube = Tube()
lap0 = tube.length / 2 - tube.rod_overlap / 2
lap1 = tube.length / 2 + tube.rod_overlap / 2

print(f"ellipse {2*tube.a:.0f} x {2*tube.b:.0f} mm, wall {tube.wall_t} mm x 1, "
      f"{tube.length:.0f} mm long")
print(f"rods    {len(tube.rod_runs())} off d{tube.rod_d:.0f} at x = +/-{tube.rod_x:.0f} mm, "
      f"y = +/-{tube.rod_y:.2f} mm")
print(f"        covered from z = {min(z0 for z0, _ in tube.rod_runs()):.0f} to "
      f"{max(z1 for _, z1 in tube.rod_runs()):.0f} mm; "
      f"lap {lap0:.0f} - {lap1:.0f} mm")
print(f"        {tube.n_rods_at(100):.0f} rods in a plain section, "
      f"{tube.n_rods_at(250):.0f} in the lap")
print(f"bells   {tube.bell_area:.0f} mm^2 each, one per rod line")

ellipse 100 x 30 mm, wall 0.42 mm x 1, 500 mm long
rods    8 off d3 at x = +/-20 mm, y = +/-9.75 mm
        covered from z = 25 to 475 mm; lap 225 - 275 mm
        4 rods in a plain section, 8 in the lap
bells   10 mm^2 each, one per rod line


## The section, from `beam.py`

Copied from `structures/wing-assembler/beam.py` — the transformed-section core and nothing
else. The materials differ, so there is no single $I$: we sum $E_i I_i$ about the
$E$-weighted neutral axis.

$$ EA=\sum E_i A_i \qquad EQ=\sum E_i\!\int\! y\,dA \qquad EI_0=\sum E_i\!\int\! y^2 dA $$
$$ y_{na}=\frac{EQ}{EA} \qquad EI = EI_0 - \frac{EQ^2}{EA} $$

Every part is a signed term, so the split at the end tells you who is actually carrying the
bending. The infill gets the Gibson-Ashby knock-down, $E^{*}=E_s\rho^{2}$.

In [2]:
def _poly(pts):
    """(A, int y dA, int y^2 dA) of a closed polygon, orientation-independent."""
    x = np.asarray([p[0] for p in pts] + [pts[0][0]])
    y = np.asarray([p[1] for p in pts] + [pts[0][1]])
    cr = x[:-1] * y[1:] - x[1:] * y[:-1]
    A = cr.sum() / 2
    Q = (cr * (y[:-1] + y[1:])).sum() / 6
    I = (cr * (y[:-1] ** 2 + y[:-1] * y[1:] + y[1:] ** 2)).sum() / 12
    s = 1.0 if A >= 0 else -1.0
    return A * s, Q * s, I * s


def _disc(r, yc):
    A = math.pi * r * r
    return A, A * yc, A * (yc * yc + r * r / 4)


def bell_envelope(t: Tube, x_rod: float, y_rod: float, step: float = 0.01):
    """(A, int y dA, int y^2 dA) of the bell outline around one bore.

    The `wing_builder` profile on the outward skin normal, clipped to the tube interior,
    with the bore cut out. Rasterised: there is no clipping library here.
    """
    sy = 1.0 if y_rod >= 0 else -1.0
    ys = sy * t.b * math.sqrt(1 - (x_rod / t.a) ** 2)
    nx, ny = x_rod / t.a ** 2, ys / t.b ** 2
    nn = math.hypot(nx, ny)
    nx, ny = nx / nn, ny / nn
    ox, oy = x_rod + nx * t.bell_offset, y_rod + ny * t.bell_offset

    reach = t.bell_height + abs(t.bell_offset) + t.bell_width
    X, Y = np.meshgrid(np.arange(x_rod - reach, x_rod + reach, step),
                       np.arange(y_rod - reach, y_rod + reach, step))
    lx = (X - ox) * ny - (Y - oy) * nx          # across the bell
    ly = (X - ox) * nx + (Y - oy) * ny          # along the normal, toward the skin
    inside = ((np.abs(lx) <= t.bell_width / 2) & (ly <= t.bell_height)
              & (ly >= t.bell_height * np.abs(2 * lx / t.bell_width) ** t.bell_shape)
              & ((X / (t.a - t.wall_t)) ** 2 + (Y / (t.b - t.wall_t)) ** 2 <= 1)
              & ((X - x_rod) ** 2 + (Y - y_rod) ** 2 > t.r_bore ** 2))
    y, dA = Y[inside], step * step
    return y.size * dA, y.sum() * dA, (y * y).sum() * dA


def section(t: Tube, m, n_rods: int = 4) -> dict:
    """EI about the E-weighted neutral axis, split by part.

    skin   = the solid wall, outer profile minus the inward offset
    infill = the interior at its knocked-down modulus, minus the rod bores
    rods   = the rods themselves, `n_rods` of the four lines present
    bells  = `bell_area` of plastic at the wall modulus, spread over the bell outline
             (which displaces the infill); one per rod line whatever `n_rods` is
    """
    E_in = m.E_infill
    terms = [("skin",    m.E, *_poly(t.outer())),
             ("skin",   -m.E, *_poly(t.inner())),
             ("infill", E_in, *_poly(t.inner()))]
    mult = n_rods / len(t.rod_centres())
    for _, y in t.rod_centres():
        terms.append(("infill", -E_in * mult, *_disc(t.r_bore, y)))
        terms.append(("rods", m.E_rod * mult, *_disc(t.r_rod, y)))
    for x, y in t.rod_centres():
        A, Q, I = bell_envelope(t, x, y)
        k = t.bell_area / A
        terms.append(("infill", -E_in, A, Q, I))
        terms.append(("bells", m.E * k, A, Q, I))

    EA = sum(E * A for _, E, A, _, _ in terms)
    EQ = sum(E * Q for _, E, _, Q, _ in terms)
    y_na = EQ / EA
    parts = {}
    for name, E, A, Q, I in terms:
        parts[name] = parts.get(name, 0.0) + E * (I - 2 * y_na * Q + y_na * y_na * A)
    return dict(EI=sum(parts.values()), y_na=y_na, parts=parts)

## Three cases for the plastic

The tube was printed standing on end, so the wall's modulus **along the beam** is across
layer lines and we do not know it. Three brackets:

| case | wall $E$ | what it assumes |
|---|---|---|
| **A** | 0 | the plastic transfers shear and carries no direct stress at all — the boom idealization, the floor |
| **B** | 2000 MPa | printed PLA across layers, the deliberately low number used for the wing |
| **C** | 3500 MPa | datasheet bulk PLA, as if the layers did not matter — the optimistic end |

The infill follows the wall in every case, at 7 % and $E^{*}=E_s\rho^2$. The bells take the
wall's modulus on their 10 mm², so in case A they vanish too.

In [3]:
@dataclass
class PLA:
    E: float = 2_000.0          # MPa  printed wall, along the beam
    infill: float = 0.07        # cubic, as printed
    infill_exp: float = 2.0     # Gibson-Ashby open cell
    E_rod: float = 125_000.0    # MPa  3 mm pultruded carbon, axial
    sigma_rod_c: float = 600.0  # MPa  working compressive allowable for the rod

    @property
    def E_infill(self) -> float:
        return self.E * self.infill ** self.infill_exp


CASES = {"A  rods only": PLA(E=0.0),
         "B  E = 2000":  PLA(E=2_000.0),
         "C  E = 3500":  PLA(E=3_500.0)}

print(f"{'case':<14}{'rods':>6}{'EI 4 rods':>13}{'EI 8 rods':>12}"
      f"{'rods':>8}{'skin':>7}{'bells':>7}{'infill':>8}")
for name, m in CASES.items():
    s4, s8 = section(tube, m, 4), section(tube, m, 8)
    sh = {k: 100 * v / s4["EI"] for k, v in s4["parts"].items()}
    print(f"{name:<14}{4:>6}{s4['EI']:>13.4e}{s8['EI']:>12.4e}"
          f"{sh['rods']:>7.1f}%{sh['skin']:>6.1f}%{sh['bells']:>6.1f}%{sh['infill']:>7.2f}%")

A, Q, I = bell_envelope(tube, tube.rod_x, tube.rod_y)
print(f"\nbell outline {A:.1f} mm^2 inside the tube, centroid y = {Q/A:.2f} mm "
      f"(rod at {tube.rod_y:.2f}); {tube.bell_area:.0f} mm^2 of it counted as plastic")

base = section(tube, CASES["A  rods only"], 4)["EI"]
for name, m in CASES.items():
    ei = section(tube, m, 4)["EI"]
    print(f"\n{name:<14} EI is {100*ei/base - 100:+5.1f} % on the rods-only floor", end="")
print()

case            rods    EI 4 rods   EI 8 rods    rods   skin  bells  infill


A  rods only       4   3.3797e+08  6.7593e+08  100.0%   0.0%   0.0%   0.00%


B  E = 2000        4   3.7300e+08  7.1093e+08   90.6%   6.7%   2.5%   0.26%


C  E = 3500        4   3.9927e+08  7.3718e+08   84.6%  10.9%   4.0%   0.42%



bell outline 40.7 mm^2 inside the tube, centroid y = 10.57 mm (rod at 9.75); 10 mm^2 of it counted as plastic



A  rods only   EI is  +0.0 % on the rods-only floor


B  E = 2000    EI is +10.4 % on the rods-only floor


C  E = 3500    EI is +18.1 % on the rods-only floor


## The load

Three-point bend: supports 450 mm apart with 25 mm of overhang each end, 80 kgf at
mid-span. Shear is $\pm P/2$ between the supports and the moment is the usual triangle,
peaking at $PL/4$.

From there it is the same three lines as the wing analysis, station by station:

$$ \kappa = \frac{M}{EI} \qquad \sigma_{rod} = E_{rod}\,(y_{rod}-y_{na})\,\kappa
   \qquad \delta = \iint \kappa $$

with $\delta = 0$ at both supports rather than a fixed root. The moment peaks at mid-span
but $EI$ nearly doubles there, because the rods lap — so the worst station is the **edge of
the lap**, not the load point.

In [4]:
P_TEST = 80.0 * G          # N, the load it held
SUPPORT_SPAN = 450.0       # mm


def _cum(f, x):
    """Cumulative trapezoid from x[0]."""
    return np.concatenate([[0.0], np.cumsum(np.diff(x) * (f[1:] + f[:-1]) / 2)])


def three_point(t: Tube, m, P: float = P_TEST, L: float = SUPPORT_SPAN, n_pts: int = 901):
    """Bend the tube between its supports, station by station."""
    a = (t.length - L) / 2
    z = np.linspace(a, t.length - a, n_pts)
    V = np.where(z < t.length / 2, P / 2, -P / 2)
    M = P / 2 * (np.minimum(z, t.length - z) - a)

    nr = np.array([t.n_rods_at(zi) for zi in z])
    cache = {n: section(t, m, int(n)) for n in sorted(set(nr.tolist()))}
    EI = np.array([cache[n]["EI"] for n in nr])

    kappa = M / EI
    sigma_rod = m.E_rod * t.rod_y * kappa       # y_na = 0 by symmetry
    sigma_skin = m.E * t.b * kappa

    delta = _cum(_cum(kappa, z), z)             # zero slope-free, then level the supports
    delta -= delta[-1] * (z - z[0]) / (z[-1] - z[0])

    k = int(np.argmax(sigma_rod))
    return dict(z=z, V=V, M=M, EI=EI, n=nr, kappa=kappa, sigma_rod=sigma_rod,
                sigma_skin=sigma_skin, delta=delta, k=k)


res = {name: three_point(tube, m) for name, m in CASES.items()}

print(f"80 kgf = {P_TEST:.0f} N at mid-span, supports {SUPPORT_SPAN:.0f} mm apart, "
      f"peak moment {P_TEST * SUPPORT_SPAN / 4 / 1000:.1f} N.m\n")
print(f"{'case':<14}{'peak rod':>10}{'at z':>7}{'at mid':>9}{'skin':>8}"
      f"{'mid defl':>11}{'x600 MPa':>11}")
for name, r in res.items():
    m = CASES[name]
    mid = len(r["z"]) // 2
    peak = r["sigma_rod"][r["k"]]
    print(f"{name:<14}{peak:>9.1f} {r['z'][r['k']]:>6.0f} "
          f"{r['sigma_rod'][mid]:>8.1f} {r['sigma_skin'][mid]:>7.2f} "
          f"{abs(r['delta'][mid]):>10.2f} {m.sigma_rod_c / peak:>10.2f}")
print("\nMPa, mm; 'x600 MPa' is what 80 kgf scales by before a rod reaches "
      "the 600 MPa working allowable.")

80 kgf = 785 N at mid-span, supports 450 mm apart, peak moment 88.3 N.m

case            peak rod   at z   at mid    skin   mid defl   x600 MPa
A  rods only      282.3    224    159.2    0.00       3.75       2.13
B  E = 2000       255.8    224    151.4    3.73       3.42       2.35
C  E = 3500       239.0    224    146.0    6.29       3.22       2.51

MPa, mm; 'x600 MPa' is what 80 kgf scales by before a rod reaches the 600 MPa working allowable.


## What 80 kgf tells us

**The rods do the bending, and the unknown does not matter much.** Between "the plastic
contributes nothing" and "the plastic is bulk PLA" there is 18 % of $EI$ and 15 % of rod
stress. That spread is smaller than the uncertainty in almost everything else here, so
there was no need to settle the wall's modulus before getting a usable number — the rods
are 85–100 % of the section whichever way it goes.

**The 7 % infill is not a structural element.** It is 0.3–0.4 % of $EI$. Its job is to hold
the wall against buckling and to keep the bores where they belong, not to carry moment.
This test also cannot say whether it does that job: the bell around each bore reaches the
skin, so the rod hands its load straight to the wall and never loads the fill.

**The bells are worth 2.5–4 % of $EI$.** Four bells of 10 mm² each carry less bending than
the 0.42 mm skin, because the skin runs the full perimeter out to $y = \pm 15$ mm. Each bell
sits at $y = 10.6$ mm, just outside its rod. Per mm² it is placed about as well as a rod, but
at 1/35 to 1/60 of the modulus. Its job is the shear path, not the moment. Counting it lowers the
peak rod stress by 2.4 % (case B) to 4 % (case C), and does nothing in case A.

**It puts a hard floor under the installed rod allowable: 282 MPa.** The article held
80 kgf, and the most conservative reading of the section — case A, plastic doing nothing —
says the worst rod was at 282 MPa when it did. Anything less than that as an allowable is
already contradicted. It is 47 % of the 600 MPa we assume, so the test neither confirms nor
threatens that assumption; it just moves the lower bracket up.

**The worst station is the edge of the lap, not the load point.** The moment peaks at
mid-span, but the rods overlap there and $EI$ doubles, so rod stress drops to 151 MPa under
the load and peaks at 256 MPa at z = 224 mm where the lap ends. The later impact broke the
rods in compression *just outside the joints* — which is where this puts them. One data
point, but it is the right one.

**Capacity.** 80 kgf scales by 2.1 to 2.5 before a rod reaches 600 MPa: **170 to 200 kgf**
on this article, in this fixture. Predicted mid-span deflection at 80 kgf is 3.2–3.8 mm,
under 1 % of the support span. Deflection was not measured, so that number is a standing
prediction rather than a check — worth measuring if the test is ever re-run.

The joint sleeve is still left out. It only adds stiffness, so leaving it out keeps the
stress numbers on the safe side.